# Support Vector Machines (SVM)

**Support Vector Machines (SVM)** are among the most powerful and versatile supervised learning algorithms in machine learning. Unlike Logistic Regression which focuses on modeling class probabilities, SVM focuses on finding the **optimal decision boundary** that separates classes with the **maximum possible margin**.

In these notes, we will cover:
1. **Geometric Intuition:** Margins, decision boundaries, and support vectors.
2. **Hard Margin SVM:** Full mathematical derivation, constraint optimization, and margin maximization.
3. **Soft Margin SVM:** Slack variables ($\xi$), the regularization hyperparameter $C$, and handling noisy data.
4. **The Kernel Trick:** Geometric intuition, feature mapping to higher dimensions, and kernel functions.
5. **Kernel Types:** Linear, RBF (Radial Basis Function), and Polynomial kernels with formulas.
6. **Advantages & Limitations of SVM.**
7. **Python Implementations:** Hands-on Scikit-Learn code with decision boundary visualizations.

## 1. SVM Geometric Intuition

### The Objective: Finding the Widest Street
Imagine you have two classes of data points plotted on a 2D plane. Many possible straight lines could separate these two classes. **Which line is the best?**

- **Logistic Regression** would find a boundary that maximizes the likelihood of the data.
- **SVM** takes a fundamentally different approach: it finds the boundary that creates the **widest possible gap (margin)** between the two classes.

Think of the decision boundary as a road, and the data points of each class as buildings on either side. SVM finds the **widest road** that can be built between the two groups of buildings. This wide road ensures that even if new buildings (unseen data) are constructed slightly off from the original ones, they will still be on the correct side of the road.

### Why Maximize the Margin?
A larger margin means the classifier is more **robust** and **generalizes better** to unseen data. A boundary that barely separates the classes (thin margin) is sensitive to noise and small perturbations, while a wide margin provides a comfortable buffer zone.

### Mathematically Defining the Decision Boundary

The decision boundary (hyperplane) in $d$-dimensional space is defined by:

$$\vec{w} \cdot \vec{x} + b = 0$$

Where:
- $\vec{w}$ = Weight vector (perpendicular/normal to the hyperplane). It defines the **orientation** of the boundary.
- $\vec{x}$ = Input feature vector.
- $b$ = Bias term. It defines the **position** (offset) of the boundary from the origin.

### The Margin Boundaries
To create a margin around the decision boundary, we define two parallel hyperplanes ("gutters" or "street edges"):

$$\vec{w} \cdot \vec{x} + b = +1 \quad \text{(Positive boundary)}$$
$$\vec{w} \cdot \vec{x} + b = -1 \quad \text{(Negative boundary)}$$

- All positive class points should satisfy: $\vec{w} \cdot \vec{x} + b \ge +1$
- All negative class points should satisfy: $\vec{w} \cdot \vec{x} + b \le -1$

The region between $+1$ and $-1$ is the **margin zone** (the "street") — no data points should fall inside this zone in a Hard Margin SVM.

### Support Vectors

**Support Vectors** are the data points that lie **exactly on the margin boundaries** ($\vec{w} \cdot \vec{x} + b = +1$ or $\vec{w} \cdot \vec{x} + b = -1$).

#### Why are they critical?
- These are the **only points** that influence the position and orientation of the decision boundary.
- If you move or remove a support vector, the decision boundary **changes**.
- If you move or remove any **non-support-vector** point (as long as it stays on the correct side of the margin), the boundary **does not change at all**.
- This makes SVM highly **memory-efficient**: the entire model is defined by just a few critical data points, not the entire training set.

#### Key Insight
The name "Support Vector Machine" comes directly from this concept — the machine (model) is entirely supported (defined) by these vectors (data points).

## 2. Hard Margin SVM: Mathematical Derivation

### Constraint Formulation
For a binary classification problem with labels $y_i \in \{-1, +1\}$, we need every data point to be correctly classified **outside** the margin. This is expressed as a single unified constraint:

$$y_i (\vec{w} \cdot \vec{x}_i + b) \ge 1, \quad \forall i = 1, 2, \dots, n$$

Let's verify this works for both classes:
- If $y_i = +1$ (positive class): $\vec{w} \cdot \vec{x}_i + b \ge +1$ ✓
- If $y_i = -1$ (negative class): $\vec{w} \cdot \vec{x}_i + b \le -1$ ✓ (multiplying both sides by $-1$ flips the inequality)

### Computing the Margin Width
The perpendicular distance from a point $\vec{x}_0$ to the hyperplane $\vec{w} \cdot \vec{x} + b = 0$ is:

$$\text{distance} = \frac{|\vec{w} \cdot \vec{x}_0 + b|}{\|\vec{w}\|}$$

The margin is the distance between the two boundary hyperplanes. Since the positive boundary has $\vec{w} \cdot \vec{x} + b = +1$ and the negative boundary has $\vec{w} \cdot \vec{x} + b = -1$:

$$\text{Margin} = \frac{1}{\|\vec{w}\|} + \frac{1}{\|\vec{w}\|} = \frac{2}{\|\vec{w}\|}$$

### The Optimization Problem
We want to **maximize** the margin $\frac{2}{\|\vec{w}\|}$, which is equivalent to **minimizing** $\|\vec{w}\|$, which is equivalent to minimizing $\frac{1}{2} \|\vec{w}\|^2$ (for mathematical convenience — the squared form is differentiable and the $\frac{1}{2}$ simplifies the derivative).

$$\boxed{\text{Minimize: } \frac{1}{2} \|\vec{w}\|^2}$$
$$\text{Subject to: } y_i (\vec{w} \cdot \vec{x}_i + b) \ge 1, \quad \forall i$$

This is a **Quadratic Programming (QP)** problem — a convex optimization with a quadratic objective and linear constraints. It is guaranteed to have a unique global minimum.

#### Solving: Lagrangian Dual
This constrained optimization is typically solved using **Lagrange Multipliers** ($\alpha_i \ge 0$) and converting to the **dual form**, which only depends on the dot products $\vec{x}_i \cdot \vec{x}_j$ between training points. This dot-product formulation is what later enables the **Kernel Trick**.

### Limitation of Hard Margin SVM
Hard Margin SVM requires **perfect linear separability**. If even a single outlier or noisy point crosses the boundary, the optimization problem becomes **infeasible** (no valid margin exists). This makes it impractical for real-world noisy datasets.

## 3. Soft Margin SVM

### The Need for Flexibility
Real-world datasets are rarely perfectly linearly separable. They contain noise, outliers, and overlapping class distributions. Hard Margin SVM fails in all these scenarios. **Soft Margin SVM** introduces controlled flexibility by allowing some data points to violate the margin or even be misclassified.

### Slack Variables ($\xi_i$)
For each data point $i$, we introduce a **slack variable** $\xi_i \ge 0$ that quantifies how much the point violates the margin constraint:

$$y_i (\vec{w} \cdot \vec{x}_i + b) \ge 1 - \xi_i$$

The slack variable captures three scenarios:

| Condition | $\xi_i$ Value | Meaning |
| :--- | :---: | :--- |
| Point is correctly classified and **outside** the margin | $\xi_i = 0$ | No violation. The point is in the safe zone. |
| Point is correctly classified but **inside** the margin | $0 < \xi_i < 1$ | Margin violation, but correct side. |
| Point is **misclassified** (wrong side of boundary) | $\xi_i \ge 1$ | Classification error. |

### The Soft Margin Optimization Problem
The new objective function balances **two competing goals**:
1. **Maximize the margin** (minimize $\|\vec{w}\|^2$).
2. **Minimize the total error** (minimize $\sum \xi_i$).

$$\boxed{\text{Minimize: } \frac{1}{2} \|\vec{w}\|^2 + C \sum_{i=1}^{n} \xi_i}$$
$$\text{Subject to: } y_i (\vec{w} \cdot \vec{x}_i + b) \ge 1 - \xi_i, \quad \xi_i \ge 0, \quad \forall i$$

### The Hyperparameter $C$
The parameter $C$ controls the **trade-off between margin width and classification errors**:

| $C$ Value | Behavior | Risk |
| :--- | :--- | :--- |
| **Large $C$** (e.g., $1000$) | Penalizes errors heavily. Prioritizes correct classification of every point. Narrow margin. | **Overfitting** (approaches Hard Margin) |
| **Small $C$** (e.g., $0.01$) | Tolerates more errors. Prioritizes a wider margin for better generalization. | **Underfitting** (too many misclassifications) |
| **Optimal $C$** | Balanced trade-off between margin width and classification accuracy. | Best generalization |

### Parallel to Logistic Regression
The Soft Margin SVM loss function is conceptually similar to **Logistic Regression with L2 regularization**:
- The $\frac{1}{2} \|\vec{w}\|^2$ term acts as **L2 regularization** (controls model complexity).
- The $C \sum \xi_i$ term acts as the **classification loss** (penalizes errors).
- SVM uses **Hinge Loss** instead of Log Loss, but the regularization structure is analogous.

## 4. The Kernel Trick: Geometric Intuition

### The Problem: Non-Linearly Separable Data
Even with Soft Margin SVM, some datasets simply **cannot be effectively separated by any straight line or hyperplane**. For example:
- Concentric circles (one class inside the other).
- XOR-style patterns.
- Spiral or interleaved data clusters.

A linear boundary will always misclassify a significant portion of the data.

### The Solution: Map to a Higher Dimension
The key insight is: **data that is not linearly separable in a low-dimensional space may become linearly separable in a higher-dimensional space.**

#### Geometric Visualization
Imagine two classes of points mixed together on a 2D plane (e.g., Class A forms a circle in the center, surrounded by Class B). No straight line can separate them.

Now, apply a transformation $\phi$ that maps each 2D point $(x_1, x_2)$ into 3D space, e.g.:
$$\phi(x_1, x_2) = (x_1, x_2, x_1^2 + x_2^2)$$

In this new 3D space, the center class points are "lifted" higher (because $x_1^2 + x_2^2$ is small near the origin), while the outer class points stay low. A **flat plane** in 3D can now perfectly separate the two classes!

When we project this separating plane back to the original 2D space, it appears as a **circle** — a non-linear decision boundary.

### The "Trick" in Kernel Trick
Explicitly computing the transformation $\phi(\vec{x})$ for every data point into a very high (or even infinite) dimensional space is:
- **Computationally expensive** (exponential number of new features).
- **Memory-intensive** (storing massive transformed feature matrices).

The **Kernel Trick** provides a mathematical shortcut: instead of computing the transformation explicitly, we define a **kernel function** $K(\vec{x}_i, \vec{x}_j)$ that directly computes the dot product in the high-dimensional space **without ever transforming the data**:

$$K(\vec{x}_i, \vec{x}_j) = \phi(\vec{x}_i) \cdot \phi(\vec{x}_j)$$

Since the SVM dual formulation only requires **dot products** between data points (not the actual coordinates in the transformed space), we can substitute the kernel function directly. This is computationally efficient because it avoids the explicit feature mapping.

## 5. Types of Kernel Functions

### 1. Linear Kernel
$$K(\vec{x}_i, \vec{x}_j) = \vec{x}_i \cdot \vec{x}_j$$
- No transformation. Equivalent to standard linear SVM.
- **Use when:** Data is already linearly separable or has many features (high-dimensional sparse data like text).

### 2. Polynomial Kernel
$$K(\vec{x}_i, \vec{x}_j) = (\gamma \cdot \vec{x}_i \cdot \vec{x}_j + r)^d$$
- Maps data to a polynomial feature space of degree $d$.
- $\gamma$ controls the influence of each dot product, $r$ is a free coefficient, and $d$ is the polynomial degree.
- **Use when:** Feature interactions of a specific degree are expected to capture class separation.

### 3. RBF (Radial Basis Function) / Gaussian Kernel
$$K(\vec{x}_i, \vec{x}_j) = \exp\left( -\gamma \|\vec{x}_i - \vec{x}_j\|^2 \right)$$
- Maps data into an **infinite-dimensional** feature space.
- $\gamma$ controls the "reach" of each training example:
  - **Large $\gamma$:** Each point has a narrow influence radius (complex, jagged boundaries → overfitting risk).
  - **Small $\gamma$:** Each point has a wide influence radius (smooth, simple boundaries → underfitting risk).
- **This is the default kernel in Scikit-Learn's SVC** and the **go-to choice** for most non-linear classification tasks.

### Kernel Comparison Summary

| Kernel | Feature Space | Key Parameters | Best Use Case |
| :--- | :--- | :--- | :--- |
| **Linear** | Original | None | Linearly separable data, high-dimensional sparse data (text) |
| **Polynomial** | Polynomial of degree $d$ | $d$, $\gamma$, $r$ | Known polynomial feature interactions |
| **RBF (Gaussian)** | Infinite-dimensional | $\gamma$ | General non-linear data (default choice) |

## 6. Advantages & Limitations of SVM

### Advantages
1. **Effective in High-Dimensional Spaces:** SVMs perform exceptionally well even when the number of features exceeds the number of samples (e.g., text classification, genomics).
2. **Versatile via Kernels:** Can handle both linear and complex non-linear classification tasks through the kernel trick.
3. **Memory Efficient:** The decision boundary depends only on the **support vectors**, not all training data.
4. **Robust to Overfitting (in high dimensions):** The margin maximization objective provides strong regularization.
5. **Applicable to Classification and Regression:** SVM for regression is called **SVR (Support Vector Regression)**.

### Limitations
1. **Slow on Large Datasets:** Training complexity is $O(n^2)$ to $O(n^3)$ — impractical for datasets with millions of samples.
2. **Sensitive to Feature Scaling:** Like KNN, SVM relies on distances. Feature scaling (`StandardScaler`) is mandatory.
3. **Kernel & Hyperparameter Selection:** Choosing the right kernel and tuning $C$, $\gamma$ requires experimentation (grid search).
4. **No Probabilistic Output (by default):** SVM outputs hard class labels, not probabilities. Probability calibration (Platt scaling) adds computational cost.
5. **Difficult to Interpret:** The model (especially with non-linear kernels) acts as a black box — no straightforward feature importance.

## 7. Code Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification, make_circles, make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report

# Set visualization contexts
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context('notebook', font_scale=1.1)

import warnings
warnings.filterwarnings('ignore')

# Helper function to plot SVM decision boundaries and margins
def plot_svm_boundary(model, X, y, title, ax=None, show_margin=True):
    if ax is None:
        ax = plt.gca()
    
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.02),
                         np.arange(y_min, y_max, 0.02))
    
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.25, cmap='coolwarm')
    
    if show_margin and hasattr(model, 'decision_function'):
        Z_df = model.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
        ax.contour(xx, yy, Z_df, levels=[-1, 0, 1], linestyles=['--', '-', '--'],
                   colors=['#3498db', '#2c3e50', '#e74c3c'], linewidths=[1.5, 2.5, 1.5])
    
    ax.scatter(X[y == 0][:, 0], X[y == 0][:, 1], color='#3498db', edgecolor='k', s=50, label='Class 0')
    ax.scatter(X[y == 1][:, 0], X[y == 1][:, 1], color='#e74c3c', edgecolor='k', s=50, label='Class 1')
    
    # Highlight support vectors
    if hasattr(model, 'support_vectors_'):
        ax.scatter(model.support_vectors_[:, 0], model.support_vectors_[:, 1],
                   s=200, facecolors='none', edgecolors='#f39c12', linewidths=2.5, label='Support Vectors')
    
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)

print("Libraries imported and helper functions defined!")

## 8. Demo A: Linear SVM — Margin & Support Vectors Visualization

We will generate a linearly separable 2D dataset, train a Linear SVM, and visualize the decision boundary, margin boundaries (dashed lines at $\vec{w} \cdot \vec{x} + b = \pm 1$), and the support vectors.

In [ ]:
# Generate linearly separable data
X_lin, y_lin = make_classification(
    n_samples=100, n_features=2, n_informative=2, n_redundant=0,
    n_clusters_per_class=1, class_sep=2.5, random_state=42
)

# Scale features
scaler = StandardScaler()
X_lin_scaled = scaler.fit_transform(X_lin)

# Train Linear SVM (Hard Margin — very large C)
svm_hard = SVC(kernel='linear', C=1000.0)
svm_hard.fit(X_lin_scaled, y_lin)

# Plot
plt.figure(figsize=(10, 7))
plot_svm_boundary(svm_hard, X_lin_scaled, y_lin, 'Linear SVM: Decision Boundary & Margin')
plt.xlabel('$x_1$', fontsize=12)
plt.ylabel('$x_2$', fontsize=12)
plt.legend(frameon=True, facecolor='white', loc='best')
plt.tight_layout()
plt.show()

print(f"Number of Support Vectors: {svm_hard.n_support_}")
print(f"Support Vectors per class: Class 0 = {svm_hard.n_support_[0]}, Class 1 = {svm_hard.n_support_[1]}")
print(f"Weights (w): {svm_hard.coef_[0]}")
print(f"Bias (b):    {svm_hard.intercept_[0]:.4f}")

## 9. Demo B: Soft Margin — Effect of Hyperparameter $C$

Let's visualize how changing $C$ affects the margin width, number of support vectors, and decision boundary flexibility on a noisier dataset.

In [ ]:
# Generate slightly noisy data
X_noisy, y_noisy = make_classification(
    n_samples=150, n_features=2, n_informative=2, n_redundant=0,
    n_clusters_per_class=1, class_sep=1.0, flip_y=0.1, random_state=42
)
X_noisy_scaled = StandardScaler().fit_transform(X_noisy)

# Train with different C values
c_values = [0.01, 1.0, 1000.0]
c_labels = ['C = 0.01\n(Wide Margin, More Errors)', 'C = 1.0\n(Balanced)', 'C = 1000\n(Narrow Margin, Few Errors)']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, c_val, label in zip(axes, c_values, c_labels):
    svm_c = SVC(kernel='linear', C=c_val)
    svm_c.fit(X_noisy_scaled, y_noisy)
    plot_svm_boundary(svm_c, X_noisy_scaled, y_noisy, label, ax=ax)
    sv_count = svm_c.n_support_.sum()
    acc = accuracy_score(y_noisy, svm_c.predict(X_noisy_scaled))
    ax.set_xlabel(f'SVs: {sv_count} | Train Acc: {acc:.2f}')

plt.suptitle('Soft Margin SVM: Effect of Hyperparameter C', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 10. Demo C: The Kernel Trick — Linear vs. RBF vs. Polynomial

We will generate a **non-linearly separable** dataset (`make_circles`) and compare the performance and decision boundaries of Linear, RBF, and Polynomial kernels.

In [ ]:
# Generate non-linear data (concentric circles)
X_circles, y_circles = make_circles(n_samples=300, noise=0.1, factor=0.4, random_state=42)
X_circles_scaled = StandardScaler().fit_transform(X_circles)

# Train three kernel variants
kernels = {
    'Linear Kernel': SVC(kernel='linear', C=1.0),
    'RBF Kernel (Gaussian)': SVC(kernel='rbf', C=1.0, gamma='scale'),
    'Polynomial Kernel (degree=3)': SVC(kernel='poly', C=1.0, degree=3)
}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, model) in zip(axes, kernels.items()):
    model.fit(X_circles_scaled, y_circles)
    acc = accuracy_score(y_circles, model.predict(X_circles_scaled))
    plot_svm_boundary(model, X_circles_scaled, y_circles, f'{name}\nAccuracy: {acc:.4f}', ax=ax, show_margin=False)

plt.suptitle('Kernel Trick: Separating Concentric Circles', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 11. Demo D: RBF Kernel on Moons Dataset with $\gamma$ Tuning

Let's explore how the RBF $\gamma$ parameter controls the complexity of the decision boundary on the `make_moons` dataset.

In [ ]:
# Generate moon dataset
X_moon, y_moon = make_moons(n_samples=200, noise=0.15, random_state=42)
X_moon_scaled = StandardScaler().fit_transform(X_moon)

# Train with different gamma values
gamma_values = [0.1, 1.0, 100.0]
gamma_labels = [
    '$\\gamma = 0.1$\n(Wide Influence, Smooth)',
    '$\\gamma = 1.0$\n(Balanced)',
    '$\\gamma = 100$\n(Narrow Influence, Complex)'
]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, g_val, label in zip(axes, gamma_values, gamma_labels):
    svm_g = SVC(kernel='rbf', C=10.0, gamma=g_val)
    svm_g.fit(X_moon_scaled, y_moon)
    acc = accuracy_score(y_moon, svm_g.predict(X_moon_scaled))
    plot_svm_boundary(svm_g, X_moon_scaled, y_moon, f'{label}\nAcc: {acc:.4f}', ax=ax, show_margin=False)

plt.suptitle('RBF Kernel: Effect of $\\gamma$ on Decision Boundary Complexity', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 12. Demo E: Full Pipeline — Breast Cancer Classification

Let's apply SVM with RBF kernel on a real-world dataset (Breast Cancer Wisconsin) and evaluate performance.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import confusion_matrix

# Load dataset
cancer = load_breast_cancer()
X_bc, y_bc = cancer.data, cancer.target

# Split and scale
X_train, X_test, y_train, y_test = train_test_split(X_bc, y_bc, test_size=0.3, random_state=42)
scaler_bc = StandardScaler()
X_train_s = scaler_bc.fit_transform(X_train)
X_test_s = scaler_bc.transform(X_test)

# Train SVM with RBF kernel
svm_bc = SVC(kernel='rbf', C=10.0, gamma='scale', random_state=42)
svm_bc.fit(X_train_s, y_train)
y_pred_bc = svm_bc.predict(X_test_s)

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_bc)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples',
            xticklabels=['Pred Malignant', 'Pred Benign'],
            yticklabels=['Actual Malignant', 'Actual Benign'])
plt.title('SVM (RBF Kernel) — Breast Cancer Confusion Matrix', fontsize=13, fontweight='bold', pad=15)
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

print(f"Test Accuracy: {accuracy_score(y_test, y_pred_bc):.4f}")
print(f"Number of Support Vectors: {svm_bc.n_support_.sum()} out of {len(X_train_s)} training points")
print()
print(classification_report(y_test, y_pred_bc, target_names=['Malignant', 'Benign']))

## Summary Checklist for Support Vector Machines

1. **Core Objective:** Find the hyperplane that maximizes the margin between two classes. The "widest street" principle.
2. **Support Vectors are Key:** Only the data points on the margin boundaries define the model. All other points are irrelevant.
3. **Hard Margin vs. Soft Margin:**
   - Hard Margin: Perfect separation required. Optimization: minimize $\frac{1}{2}\|w\|^2$ subject to $y_i(w \cdot x_i + b) \ge 1$.
   - Soft Margin: Allows errors via slack variables $\xi_i$. Optimization: minimize $\frac{1}{2}\|w\|^2 + C \sum \xi_i$.
4. **Tune $C$ Carefully:** Large $C$ = narrow margin (overfitting). Small $C$ = wide margin (underfitting).
5. **Use Kernels for Non-Linear Data:** The Kernel Trick maps data to higher dimensions without explicit transformation.
   - **RBF (default):** Best for general non-linear tasks. Tune $\gamma$ (large = overfit, small = underfit).
   - **Linear:** Use for linearly separable or high-dimensional sparse data.
   - **Polynomial:** Use when polynomial feature interactions are expected.
6. **Always Scale Features:** SVM is distance-based. Use `StandardScaler` before training.
7. **Not for Very Large Datasets:** Training is $O(n^2)$ to $O(n^3)$. For big data, consider `LinearSVC` or SGD-based approaches.